# GigShield Fraud Detection Model

**Algorithm:** Isolation Forest (unsupervised anomaly detection)

**Purpose:** Detect fraudulent claims in GigShield's zero-touch parametric insurance system. Since we don't have labeled fraud data upfront, Isolation Forest learns what "normal" claim behavior looks like and flags deviations.

**Decision Thresholds:**
- Score < 0.75 → Auto-approve
- Score 0.75 - 0.90 → Human review
- Score > 0.90 → Automatic payout hold

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

from feature_engineering import (
    generate_synthetic_zones,
    generate_synthetic_workers,
    generate_synthetic_events,
    generate_synthetic_claims,
    prepare_fraud_features,
    normalize_anomaly_scores
)

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
print('Libraries loaded successfully')

## 1. Generate Synthetic Data

In [ ]:
zones_df = generate_synthetic_zones(n_zones=50, seed=SEED)
workers_df = generate_synthetic_workers(n_workers=500, zones_df=zones_df, seed=SEED)
events_df = generate_synthetic_events(n_events=300, zones_df=zones_df, seed=SEED)
claims_df = generate_synthetic_claims(workers_df, events_df, zones_df, anomaly_rate=0.05, seed=SEED)

print(f'Claims dataset: {claims_df.shape}')
print(f'Anomalous claims: {claims_df["is_anomalous"].sum()} ({claims_df["is_anomalous"].mean()*100:.1f}%)')
claims_df.describe()

## 2. Exploratory Data Analysis

In [ ]:
feature_cols = ['claims_past_30d', 'claim_to_coverage_ratio', 'gps_distance_km', 'time_gap_hours', 'earnings_match_score']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(feature_cols):
    axes[i].hist(claims_df[col], bins=30, alpha=0.7, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=10)
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(claims_df[feature_cols].corr(), annot=True, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: GPS distance vs Time gap, colored by anomaly status
fig, ax = plt.subplots(figsize=(10, 6))
colors = claims_df['is_anomalous'].map({True: 'red', False: 'steelblue'})
ax.scatter(claims_df['gps_distance_km'], claims_df['time_gap_hours'], c=colors, alpha=0.5, s=20)
ax.set_xlabel('GPS Distance from Zone (km)')
ax.set_ylabel('Time Gap (hours)')
ax.set_title('GPS Distance vs Time Gap (Red = Anomalous)')
plt.show()

## 3. Feature Preparation

In [ ]:
X_scaled, scaler = prepare_fraud_features(claims_df)
print(f'Scaled feature matrix shape: {X_scaled.shape}')
print(f'Feature means after scaling: {X_scaled.mean(axis=0).round(4)}')
print(f'Feature stds after scaling: {X_scaled.std(axis=0).round(4)}')

## 4. Train Isolation Forest Model

In [ ]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=SEED,
    n_jobs=-1
)

model.fit(X_scaled)

# Get anomaly scores
raw_scores = model.decision_function(X_scaled)
anomaly_scores = normalize_anomaly_scores(raw_scores)
claims_df['anomaly_score'] = anomaly_scores

print(f'Anomaly score range: {anomaly_scores.min():.3f} to {anomaly_scores.max():.3f}')
print(f'Mean score: {anomaly_scores.mean():.3f}')

## 5. Threshold Analysis

In [ ]:
# Score distribution with threshold lines
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(anomaly_scores, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(x=0.75, color='orange', linestyle='--', linewidth=2, label='Review threshold (0.75)')
ax.axvline(x=0.90, color='red', linestyle='--', linewidth=2, label='Hold threshold (0.90)')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Count')
ax.set_title('Anomaly Score Distribution with Decision Thresholds')
ax.legend()
plt.tight_layout()
plt.show()

# Decision counts
auto_approve = (anomaly_scores < 0.75).sum()
human_review = ((anomaly_scores >= 0.75) & (anomaly_scores < 0.90)).sum()
hold = (anomaly_scores >= 0.90).sum()
print(f'Auto-approve: {auto_approve} ({auto_approve/len(anomaly_scores)*100:.1f}%)')
print(f'Human review: {human_review} ({human_review/len(anomaly_scores)*100:.1f}%)')
print(f'Payout hold:  {hold} ({hold/len(anomaly_scores)*100:.1f}%)')

In [ ]:
# Confusion matrix against known anomaly labels
predicted_anomaly = anomaly_scores >= 0.75
actual_anomaly = claims_df['is_anomalous'].values

print('Classification Report (threshold=0.75):')
print(classification_report(actual_anomaly, predicted_anomaly, target_names=['Normal', 'Anomalous']))

cm = confusion_matrix(actual_anomaly, predicted_anomaly)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Flagged'], yticklabels=['Normal', 'Anomalous'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. SHAP Explainability

In [ ]:
try:
    import shap
    
    explainer = shap.Explainer(model.predict, X_scaled[:100])
    shap_values = explainer(X_scaled[:100])
    
    # Summary plot
    shap.summary_plot(shap_values, X_scaled[:100], feature_names=feature_cols, show=True)
    
    # Example: explain a flagged record
    flagged_idx = np.where(anomaly_scores[:100] > 0.75)[0]
    if len(flagged_idx) > 0:
        print(f'\nSHAP explanation for flagged record #{flagged_idx[0]}:')
        shap.waterfall_plot(shap_values[flagged_idx[0]])
except ImportError:
    print('SHAP not installed. Install with: pip install shap')
    print('Skipping SHAP analysis — model training is still complete.')

## 7. Save Model

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/fraud_isolation_forest.pkl')
joblib.dump(scaler, 'models/fraud_scaler.pkl')

print('Model saved to models/fraud_isolation_forest.pkl')
print('Scaler saved to models/fraud_scaler.pkl')
print(f'Training samples: {X_scaled.shape[0]}')
print(f'Features: {feature_cols}')

## 8. Inference Example

In [ ]:
# Load model and score a single new claim
loaded_model = joblib.load('models/fraud_isolation_forest.pkl')
loaded_scaler = joblib.load('models/fraud_scaler.pkl')

# Normal claim example (Ravi filing after heavy rain)
normal_claim = np.array([[2, 0.25, 0.8, 0.5, 0.90]])
normal_scaled = loaded_scaler.transform(normal_claim)
normal_raw = loaded_model.decision_function(normal_scaled)[0]
normal_score = normalize_anomaly_scores(np.array([normal_raw]))[0]
print(f'Normal claim score: {normal_score:.3f} → {"Auto-approve" if normal_score < 0.75 else "Review"}')

# Suspicious claim example
suspicious_claim = np.array([[12, 1.5, 7.5, 8.0, 0.2]])
suspicious_scaled = loaded_scaler.transform(suspicious_claim)
suspicious_raw = loaded_model.decision_function(suspicious_scaled)[0]
suspicious_score = normalize_anomaly_scores(np.array([suspicious_raw]))[0]
print(f'Suspicious claim score: {suspicious_score:.3f} → {"Hold" if suspicious_score > 0.90 else "Review" if suspicious_score > 0.75 else "Approve"}')